# «Trustworthy Online Controlled Experiments»

Ноутбук к разбору книги Кохави, Тан и Сюй (2020): https://dilyakhakimova.com/shelf/kohavi-trustworthy/

Три проверки доверия к A/B-тесту: SRM χ²-критерием, симуляция подглядывания и тысяча A/A на своих данных. Код и числа наши, по описанию методов; нужны только Python и numpy. Подставь свои числа и запусти.

## Проверка SRM χ²-критерием

Подставь в n_a и n_b числа из своего теста. Если сплит не 50/50, задай долю контроля: например, при 90/10 — srm_check(n_a, n_b, share_a=0.9).

In [ ]:
import numpy as np
from math import erfc, sqrt

def srm_check(n_a, n_b, share_a=0.5):
    """χ²-критерий: совпадает ли соотношение групп с задуманным сплитом."""
    total = n_a + n_b
    exp_a, exp_b = total * share_a, total * (1 - share_a)
    chi2 = (n_a - exp_a) ** 2 / exp_a + (n_b - exp_b) ** 2 / exp_b
    p = erfc(sqrt(chi2 / 2))   # p-value для χ² с одной степенью свободы
    return chi2, p

# сплит 50/50, в группе B антифрод отрезал часть пользователей
n_a, n_b = 500_000, 493_000   # ← подставь свои числа
chi2, p = srm_check(n_a, n_b)
print(f"A = {n_a}, B = {n_b}, доля A = {n_a / (n_a + n_b):.4f}")
print(f"χ² = {chi2:.1f}, p-value = {p:.2g}")
print("SRM: отчёту не верим, ищем причину" if p < 0.0005 else "перекос в пределах шума")

## Симуляция подглядывания: 1000 A/A-тестов

Эффекта нет ни в одном тесте. Сравни, сколько «побед» дают проверка в конце и проверка каждый день. Поменяй days — посмотри, как растут ложные победы.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
tests, days, per_day, conv = 1000, 20, 500, 0.10   # 1000 A/A-тестов по 20 дней, по 500 пользователей в день в группе

# накопленные заказы в группах A и B; эффекта нет: конверсия одна и та же
a = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
b = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
n = per_day * np.arange(1, days + 1)

pool = (a + b) / (2 * n)
se = np.sqrt(pool * (1 - pool) * 2 / n)
z = np.abs(b - a) / n / se
significant = z > 1.96          # «значимо» при α = 0.05

print(f"смотрим один раз, в конце:          {significant[:, -1].mean():.1%} ложных побед")
print(f"смотрим каждый день, стоп при «значимо»: {significant.any(axis=1).mean():.1%} ложных побед")

## Тысяча A/A на своих данных и гистограмма p-value

Заготовка для своих данных: метрика за прошлую неделю, одно число на пользователя. Файла с данными пока нет, поэтому в примере симулированные данные. У честного расчёта p < 0.05 около 5%, гистограмма p-value — равномерное распределение, и тест Колмогорова — Смирнова это подтверждает.

In [ ]:
import numpy as np
from math import erfc, sqrt, exp

rng = np.random.default_rng(7)

# Данные: метрика за прошлую неделю, одно число на пользователя.
# Файла с данными пока нет, поэтому берём симулированные: выручка с тяжёлым хвостом, 20 000 пользователей.
# Когда будет файл — раскомментируй две строки ниже и закомментируй симуляцию:
# import pandas as pd
# values = pd.read_csv("metric_last_week.csv")["value"].to_numpy()   # ← свой файл и своя колонка
values = rng.lognormal(mean=6, sigma=1.2, size=20_000)

def p_value(a, b):
    """Двусторонний тест разницы средних; дисперсия — по пользователям."""
    se = sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    return erfc(abs(b.mean() - a.mean()) / se / sqrt(2))

# 1000 раз случайно делим пользователей пополам, как сделала бы платформа
runs = 1000
pv = np.empty(runs)
for i in range(runs):
    in_test = rng.random(len(values)) < 0.5
    pv[i] = p_value(values[~in_test], values[in_test])

print(f"p < 0.05 у {np.mean(pv < 0.05):.1%} A/A — при честной платформе около 5%\n")

# гистограмма p-value: 20 столбиков по 0.05, у равномерного распределения каждый около 5%
counts, _ = np.histogram(pv, bins=20, range=(0, 1))
for k, c in enumerate(counts):
    print(f"{k * 0.05:.2f}–{(k + 1) * 0.05:.2f}  {'█' * round(c / runs * 400)} {c / runs:.1%}")

# тест Колмогорова — Смирнова: похоже ли распределение p-value на равномерное
x = np.sort(pv)
d = max(np.max(np.arange(1, runs + 1) / runs - x), np.max(x - np.arange(runs) / runs))
lam = (sqrt(runs) + 0.12 + 0.11 / sqrt(runs)) * d
p_ks = min(1.0, 2 * sum((-1) ** (k - 1) * exp(-2 * k * k * lam * lam) for k in range(1, 101)))
print(f"\nКолмогоров — Смирнов: D = {d:.3f}, p = {p_ks:.3f}",
      "— похоже на равномерное" if p_ks > 0.05 else "— не похоже на равномерное: ищи ошибку в расчёте")